# 📦 WhatsApp Chat Archive — indexer

Run this whenever you want the archive viewer to show your latest chats
(e.g. once a week, and **always before switching phones**).
Your photos and videos are already backed up every night by FolderSync — this notebook only
rebuilds the *index* (which file belongs to which chat, and when).

**How to run on your phone:** tap ▶ on each cell from top to bottom (or *Runtime → Run all*).
Takes ~2 min for text, plus a few minutes per few thousand photos the first time (cached afterwards).

In [ ]:
#@title 1. Settings (change only if your folders are named differently)
ARCHIVE_FOLDER = "WhatsAppArchive"   #@param {type:"string"}
REPO = "https://github.com/Kandarp-Joshi-007/whatsapp-backup"  #@param {type:"string"}
COUNTRY_CODE = "91"                  #@param {type:"string"}
TIMEZONE = "Asia/Kolkata"            #@param {type:"string"}

DRIVE = "/content/drive/MyDrive"
ROOT = f"{DRIVE}/{ARCHIVE_FOLDER}"
RAW = f"{ROOT}/raw"          # FolderSync uploads the phone's WhatsApp folder here
INDEX = f"{ROOT}/index"      # the viewer reads from here

In [ ]:
#@title 2. Connect Google Drive and install the indexer
from google.colab import drive, auth
drive.mount("/content/drive")
auth.authenticate_user()
!pip -q install "git+{REPO}#subdirectory=indexer" google-api-python-client
import os
assert os.path.isdir(f"{RAW}/Databases"), f"Can't find {RAW}/Databases. Has FolderSync uploaded the WhatsApp folder yet?"
print("✅ Drive connected")

In [ ]:
#@title 3. Your 64-digit key (never saved to Drive)
# Tip: store it once in Colab ▸ 🔑 Secrets as WA_KEY so you don't have to paste it every time.
import getpass
try:
    from google.colab import userdata
    WA_KEY = userdata.get("WA_KEY")
except Exception:
    WA_KEY = None
if not WA_KEY:
    WA_KEY = getpass.getpass("Paste your 64-digit WhatsApp backup key: ")
from wa_archive.crypt15 import parse_key
parse_key(WA_KEY)
print("✅ Key looks valid (64 hex digits)")

In [ ]:
#@title 4. Decrypt the newest backup
import glob, os
from wa_archive.crypt15 import decrypt_file
backups = sorted(glob.glob(f"{RAW}/Databases/msgstore*.crypt15"), key=os.path.getmtime)
assert backups, "No msgstore*.crypt15 found. In WhatsApp: Settings > Chats > Chat backup > Back up, then let FolderSync upload."
latest = backups[-1]
print("Using", os.path.basename(latest), "from", __import__("datetime").datetime.fromtimestamp(os.path.getmtime(latest)))
decrypt_file(latest, WA_KEY, "/content/msgstore.db")   # stays on this temporary Colab machine
print("✅ Decrypted")

In [ ]:
#@title 5. Find every backed-up media file in Drive (lets the viewer open them instantly)
import json
from googleapiclient.discovery import build
from wa_archive.drive_ids import folder_id_for_path, list_tree
svc = build("drive", "v3", cache_discovery=False)
ids = list_tree(svc, folder_id_for_path(svc, f"{ARCHIVE_FOLDER}/raw"))
json.dump(ids, open("/content/drive_ids.json", "w"))
print(f"✅ {len(ids):,} files found in Drive")

In [ ]:
#@title 6. Build the archive index
import glob, os, shutil
from wa_archive.msgstore import read_msgstore
from wa_archive.contacts import load_contacts
from wa_archive.build import build_index

contact_files = [p for p in glob.glob(f"{ROOT}/contacts*") + glob.glob(f"{ROOT}/aliases.json")]
names = load_contacts(contact_files, COUNTRY_CODE) if contact_files else {}
print(f"{len(names)} contact names loaded from {[os.path.basename(p) for p in contact_files] or 'nothing (chats will show phone numbers)'}")

cache_drive, cache_local = f"{ROOT}/thumbcache.sqlite", "/content/thumbcache.sqlite"
if os.path.exists(cache_drive): shutil.copy(cache_drive, cache_local)

chats = read_msgstore("/content/msgstore.db")
build_index(chats, "/content/index", names=names, media_root=RAW,
            drive_ids=json.load(open("/content/drive_ids.json")), tz=TIMEZONE, thumb_cache=cache_local)
shutil.copy(cache_local, cache_drive)

In [ ]:
#@title 7. Upload the index to Drive
import os, shutil, filecmp
os.makedirs(INDEX, exist_ok=True)
new = set(os.listdir("/content/index"))
changed = 0
for f in new:
    src, dst = f"/content/index/{f}", f"{INDEX}/{f}"
    if not os.path.exists(dst) or not filecmp.cmp(src, dst, shallow=False):
        shutil.copy(src, dst); changed += 1
for f in os.listdir(INDEX):
    if f not in new: os.remove(f"{INDEX}/{f}")
drive.flush_and_unmount()
os.remove("/content/msgstore.db")
print(f"✅ Done — {changed} files updated. Close and reopen the Chat Archive app to see the update.")